# Multiple regression and correlation for clinicians
### A worked example with NASA open data: the Inspiration4 civilian crew

**Author:** Víctor Junior Godínez-Vázquez · Teaching material · License CC BY 4.0 (text) / MIT (code) · *Versión en español: `clase_regresion_nasa.ipynb`*

**How to use this notebook:** menu **Runtime → Run all** (or `Ctrl + F9`). Nothing to install.

**Clinical question:** do glucose, potassium, creatinine and total protein change with a 3-day orbital flight, comparing each crew member with themselves?

**What you will learn**
1. A categorical variable (flight phase) becomes *indicator (dummy) variables*.
2. What changes when each person is measured several times (**repeated measures**).
3. Comparing three models: **A** naive, **B** multiple regression with the crew member, **C** mixed model.
4. Checking data quality **before** analysing, and checking assumptions with **residuals**.

> **Caution:** this is an **exploratory, teaching** analysis. These four laboratory tests were not part of the original study's preregistered analysis (doi:10.17605/OSF.IO/W3NP2). With 4 people, no result should be used for clinical decisions.

**Data:** NASA Open Science Data Repository, study OSD-575 (doi:10.26030/mc5d-p710). Derived file `i4_analysis.csv`, available at https://osf.io/57cdk

## 1. Set up
We load the libraries. `statsmodels` is Python's library for statistical models (the equivalent of SPSS menus).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
print("Ready ✔")

## 2. Load the data
The notebook tries to download the file from the GitHub repository. If it cannot, it will ask you to upload `i4_analysis.csv` from your computer.

In [ ]:
BASE = "https://raw.githubusercontent.com/ViiJX-stack/regression-multiple-clinicos/main/"
d = None
for path in ["i4_analysis.csv", "datos/i4_analysis.csv"]:
    try:
        d = pd.read_csv(BASE + path)
        print("Data downloaded from GitHub ✔")
        break
    except Exception:
        pass
if d is None:
    from google.colab import files
    print("Download failed. Please upload i4_analysis.csv:")
    uploaded = files.upload()
    d = pd.read_csv(list(uploaded.keys())[0])

# The numeric prefix fixes the order; 'pre' is the reference category
d["phase"] = d["phase"].map({"1_pre": "1_pre", "2_return": "2_return", "3_recovery": "3_recovery"})
d["crew_member"] = d["crew"]
ORDER = ["L-92", "L-44", "L-3", "R+1", "R+45", "R+82", "R+194"]
print(d.shape[0], "rows =", d.crew_member.nunique(), "crew members ×", d.timepoint.nunique(), "timepoints")
d[["crew_member", "timepoint", "phase", "glucose_raw", "potassium_raw", "creatinine", "total_protein"]]

**Reading the table.** Each row is one blood sample. There are **28 rows but only 4 people**: each crew member appears 7 times.
- `L-92` = 92 days before launch · `R+1` = 1 day after return.
- Units: glucose mg/dL · potassium mmol/L · creatinine mg/dL · total protein g/dL.

## 3. Data quality: check first, analyse second
Look at glucose at **R+45**. Values < 55 mg/dL in asymptomatic adults are not physiological: they indicate the sample was processed late and the cells **consumed the glucose in the tube** (in-vitro glycolysis). The same delay can **falsely raise potassium** (potassium leaks out of cells).

In [ ]:
r45 = d[d.timepoint == "R+45"][["crew_member", "glucose_raw", "potassium_raw"]]
print("Glucose and potassium at R+45 (raw values):")
print(r45.to_string(index=False))
print("\nRule fixed before analysis: glucose < 55 mg/dL → exclude glucose and potassium from that draw.")
# The *_analysis columns already apply the rule (R+45 = empty)
LABS = {"glucose_analysis": "Glucose (mg/dL)", "potassium_analysis": "Potassium (mmol/L)",
        "creatinine": "Creatinine (mg/dL)", "total_protein": "Total protein (g/dL)"}
print("\nAvailable values per test:", {v: int(d[k].notna().sum()) for k, v in LABS.items()})

## 4. Look at the data: one line per crew member
Before any model, **plot**. Each colour is one person; the grey band marks the day after return.

In [ ]:
colours = {"C001": "#2a78d6", "C002": "#eb6834", "C003": "#1baf7a", "C004": "#c98500"}
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
x = np.arange(len(ORDER))
for ax, (col, name) in zip(axes.ravel(), LABS.items()):
    ax.axvspan(2.5, 3.5, color="#eeeeee")
    for c, g in d.groupby("crew_member"):
        g = g.set_index("timepoint").loc[ORDER]
        ax.plot(x, g[col], marker="o", color=colours[c], label=c)
    ax.set_xticks(x); ax.set_xticklabels(ORDER, fontsize=8); ax.set_title(name)
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Question for the group:** do the lines differ more **between people** or **between timepoints**? For example, C002's creatinine is always lower than C001's. That between-person difference is the key to this exercise.

## 5. Categorical variables → indicator (dummy) variables
Regression needs numbers. *Flight phase* has 3 categories, so it becomes **2 columns of 0/1**; the third category (**pre**) is the **reference**. SPSS does the same internally.

In [ ]:
dummies = pd.get_dummies(d["phase"], drop_first=True).astype(int)
pd.concat([d[["crew_member", "timepoint", "phase"]], dummies], axis=1).head(7)

**Interpretation:**
`y = β0 + β1·return + β2·recovery`
- **β0** = mean **before** flight.
- **β1** = difference **return − pre**.
- **β2** = difference **recovery − pre**.

## 6. Three models for the same question
| Model | Formula | What it assumes |
|---|---|---|
| **A. Naive** | `y ~ phase` | 28 independent observations (false: they are 4 people) |
| **B. Multiple regression** | `y ~ phase + crew_member` | Each person has their own baseline → compares each person with themselves |
| **C. Mixed model** | `y ~ phase`, random intercept per crew member | Like B, but treats crew members as a sample from a population |

Model **B** is a multiple regression: 2 phase dummies + 3 crew dummies = **5 independent variables**.

In [ ]:
def fit_models(col):
    data = d[["crew_member", "phase", col]].dropna().rename(columns={col: "y"})
    mA = smf.ols("y ~ C(phase)", data=data).fit()
    mB = smf.ols("y ~ C(phase) + C(crew_member)", data=data).fit()
    mC = smf.mixedlm("y ~ C(phase)", data=data, groups=data["crew_member"]).fit(reml=True)
    return data, mA, mB, mC

def row(model, name, term):
    return {"model": name, "coef": model.params[term], "SE": model.bse[term], "p": model.pvalues[term]}

results = []
models = {}
for col, name in LABS.items():
    data, mA, mB, mC = fit_models(col)
    models[col] = (data, mA, mB, mC)
    var_crew = float(mC.cov_re.iloc[0, 0]); var_res = float(mC.scale)
    icc = var_crew / (var_crew + var_res)
    for term, label in [("C(phase)[T.2_return]", "return − pre"), ("C(phase)[T.3_recovery]", "recovery − pre")]:
        for m, nm in [(mA, "A naive"), (mB, "B + crew member"), (mC, "C mixed")]:
            r = row(m, nm, term); r.update({"test": name, "contrast": label, "n": len(data)})
            r["R² (A/B)"] = getattr(m, "rsquared", np.nan); r["ICC"] = icc if nm == "C mixed" else np.nan
            results.append(r)

table = pd.DataFrame(results)[["test", "contrast", "model", "n", "coef", "SE", "p", "R² (A/B)", "ICC"]]
table.round(3)

### How to read the table
- **coef**: mean difference in the test's units (for example, mg/dL).
- **SE** (standard error): uncertainty of the coefficient. **Compare the SE of A with B and C.**
- **R²**: proportion of variability explained. Its square root is the **multiple correlation coefficient (R)**.
- **ICC** (intraclass correlation coefficient): how much of the variability is due to **differences between people**. If it is high, ignoring the person (model A) is a serious mistake.

In [ ]:
print("Summary per test (contrast return − pre):\n")
for col, name in LABS.items():
    t = table[(table["test"] == name) & (table.contrast == "return − pre")].set_index("model")
    icc = t.loc["C mixed", "ICC"]
    print(f"{name}: ICC = {icc:.2f} → {icc*100:.0f}% of the variability is due to differences between people")
    print(f"   SE model A = {t.loc['A naive','SE']:.3f} | SE model B = {t.loc['B + crew member','SE']:.3f}"
          f" | p A = {t.loc['A naive','p']:.3f} | p B = {t.loc['B + crew member','p']:.3f} | p C = {t.loc['C mixed','p']:.3f}")
    print(f"   R² A = {t.loc['A naive','R² (A/B)']:.2f} → multiple R = {np.sqrt(t.loc['A naive','R² (A/B)']):.2f} | "
          f"R² B = {t.loc['B + crew member','R² (A/B)']:.2f} → multiple R = {np.sqrt(t.loc['B + crew member','R² (A/B)']):.2f}\n")

**Key message:** when people differ a lot from each other (high ICC), model **A** mixes that variability into the error, its **SE grows** and it can **hide** a real effect. Models **B** and **C** compare each person with themselves.

**What these data show**
- **Creatinine and potassium:** ICC of 0.67 and 0.59. Adding the crew member (model B) lowers the SE by 33% to 40%, and R² for creatinine rises from 0.01 to 0.68. That is the effect of comparing each person with themselves.
- **Glucose:** rose by about 18 mg/dL the day after return, **in all 4 crew members**. Here the ICC is ≈ 0, so the three models agree. This is consistent with a stress response (counter-regulatory hormones), but **we do not know whether samples were fasting**, and some baseline values are also low (56 and 62 mg/dL), possibly from the same tube problem. It is an **exploratory** finding.
- **Multiple comparisons:** 4 tests × 2 contrasts = 8 tests. With a Bonferroni correction the threshold would be 0.05 / 8 ≈ 0.006.
- **Total protein:** no change and no differences between people (ICC ≈ 0).

**Technical note:** Python's mixed model (`statsmodels`) uses a normal approximation for *p* values. With so few groups, the original paper used the Kenward-Roger correction in R, which is more conservative. For teaching purposes the difference does not change the message.

## 7. Assumptions are checked with **residuals**, not with the variables
The normality assumption refers to the **model's errors** (residuals), not to each variable being normally distributed. We check model B for creatinine.

In [ ]:
data, mA, mB, mC = models["creatinine"]
res, fitted = mB.resid, mB.fittedvalues
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.scatter(fitted, res); a1.axhline(0, color="gray", ls="--")
a1.set_xlabel("Fitted values"); a1.set_ylabel("Residuals"); a1.set_title("Residuals vs fitted (constant variance)")
sm.qqplot(res, line="s", ax=a2); a2.set_title("Q-Q plot (normality of residuals)")
plt.tight_layout(); plt.show()

**What to look for:** on the left, a cloud with no funnel shape; on the right, points close to the line. With 28 data points, small deviations are expected; formal normality tests have little power with samples this small.

## 8. What about correlation?
Pearson's correlation measures the linear association between two variables. **Careful:** if you compute it with all 28 rows, you mix differences between people with changes within each person.

In [ ]:
cols = ["glucose_analysis", "potassium_analysis", "creatinine", "total_protein"]
print("Pearson correlation using all 28 rows (mixes between- and within-person):")
print(d[cols].corr().round(2))
centred = d[cols].sub(d.groupby("crew_member")[cols].transform("mean"))
print("\nCorrelation WITHIN each person (subtracting each crew member's mean):")
print(centred.corr().round(2))

**Example in these data:** creatinine and potassium have r = 0.45 across all 28 rows, but r ≈ 0 within each person. The "association" came from crew members with higher creatinine also tending to have higher potassium, not from both changing together within the same person. It is the same principle as models A and B: **the person is a confounder**.

## 9. The same in SPSS
- **Model B (phase + crew member):** Analyze → General Linear Model → Univariate. Dependent variable: the test; Fixed factors: phase and crew member.
- **Model C (mixed):** Analyze → Mixed Models → Linear. Subjects: crew member; Repeated: timepoint. Dependent variable: the test; Factors: phase; Random effects: intercept by crew member.

**Message:** the software does not matter; building the right model does.

## 10. Frequently asked questions
**Why not a one-way ANOVA?** That is model A: it assumes 28 independent observations. A repeated-measures ANOVA would be valid, but it requires complete data (no excluded values, as at R+45) and is less flexible than the mixed model.

**Why not a paired t-test?** It compares only two timepoints and wastes the three baseline draws. The model uses all of them.

**Isn't n = 4 too small?** Yes, for generalising, but it is **the entire crew** of the first all-civilian orbital mission. The right analysis with few people is precisely the one that compares each person with themselves.

**Does this prove the flight changed the tests?** No. There is no ground control group, and these analyses are exploratory.

## References
1. Overbey EG, et al. The Space Omics and Medical Atlas (SOMA) and international astronaut biobank. *Nature*. 2024;632:1145-54.
2. Jones CW, et al. Molecular and physiological changes in the SpaceX Inspiration4 civilian crew. *Nature*. 2024;632:1155-64.
3. Godínez-Vázquez VJ. Change in the neutrophil-to-lymphocyte ratio after a 3-day orbital flight [preprint]. *medRxiv*. 2026. doi:10.64898/2026.10.16.26365757
4. UNC Eshelman School of Pharmacy. Biostatistics, Lesson 6: Continuous outcomes, linear regression (pre-reading).
5. Swinscow TDV, Campbell MJ. Statistics at Square One. 9th ed. BMJ Publishing Group; 1997. Chapter 11: Correlation and regression. https://www.bmj.com/about-bmj/resources-readers/publications/statistics-square-one/11-correlation-and-regression (free pre-reading; current edition: Campbell MJ. Statistics at Square One. 12th ed. Wiley; 2021).
6. NASA Open Science Training Team. (2025). Open Science 101. NASA. https://science.nasa.gov/open-science/training/

*This notebook was prepared with AI assistance (Claude, Anthropic); the author reviewed the content and is responsible for it.*